## 2.2. Task Submission - City Bike New York

In [ ]:
!pip install pyarrow

### importing libraries:

In [1]:
#importing libraries
import pandas as pd
import numpy as np
import os
import requests
import json
from datetime import datetime

print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("json: OK")

pandas: 2.3.3
requests: 2.32.4
json: OK


##### Some notes:
- my computer crashed with these files. it doesn't have enough RAM. So I proceeded with Kaggle, which offers me 30gb instead of my original 8gb.
- Some constrictions with CPU were found, but manageable: Had to discard three columns, ride_id, start_station_id e end_station_id.

### uploading and checking citibike files:

In [2]:
# checking columns available in one CSV file
sample_path = '/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202201-citibike-tripdata/202201-citibike-tripdata_1.csv'
df_sample = pd.read_csv(sample_path, nrows=1)
print(df_sample.columns.tolist())

['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']


Existent columns:
ride_id,
rideable_type,
started_at,
ended_at,
start_station_name,
start_station_id,
end_station_name,
end_station_id,
start_lat,
start_lng,
end_lat,
end_lng,
member_casual

In [4]:
#this is a file explorer: it runs all folders and sub folders and prints the complete path for each file.
for root, dirs, files in os.walk('/kaggle/input/datasets/danielabranca/full-data-set'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202201-citibike-tripdata/202201-citibike-tripdata_2.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202201-citibike-tripdata/202201-citibike-tripdata_1.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202205-citibike-tripdata/202205-citibike-tripdata_1.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202205-citibike-tripdata/202205-citibike-tripdata_3.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202205-citibike-tripdata/202205-citibike-tripdata_2.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202209-citibike-tripdata/202209-citibike-tripdata_3.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202209-citibike-tripdata/202209-citibike-tripdata_4.csv
/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202209-citibike-

In [5]:
#By uploading a zip folder to Kaggle, Kaggle will unzip them in the process. Therefore, the following code applies:

#defining the path/folder with all csvs:
folderpath = '/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata'

#defining the columns that will be uploaded (due to RAM constrictions):
cols = ['rideable_type', 'started_at', 'ended_at', 'start_station_name', 
        'end_station_name', 'start_lat', 'start_lng', 
        'end_lat', 'end_lng', 'member_casual']

#creating an empty dataframe that will grow as files are added:
df = pd.DataFrame()

#it runs over all folders and sub folders and filters only csv files:
for root, dirs, files in os.walk(folderpath):
    for file in files:
        if file.endswith('.csv'):
            filepath = os.path.join(root, file) #it builds the complete path for each file
            df_temp = pd.read_csv(filepath, usecols=cols) #it reads the csv reading only the selected columns
            df = pd.concat([df, df_temp], ignore_index=True) #it adds the new file to the main dataframe
            del df_temp #it deletes the temporary dataframe to release memory immediately (useful when there's CPU constrictions)
            print(f"Read: {file}") #it confirms the file was read.

print(f"\nTotal of lines: {len(df)}")

Read: 202201-citibike-tripdata_2.csv
Read: 202201-citibike-tripdata_1.csv
Read: 202205-citibike-tripdata_1.csv
Read: 202205-citibike-tripdata_3.csv
Read: 202205-citibike-tripdata_2.csv
Read: 202209-citibike-tripdata_3.csv
Read: 202209-citibike-tripdata_4.csv
Read: 202209-citibike-tripdata_1.csv
Read: 202209-citibike-tripdata_2.csv
Read: 202203-citibike-tripdata_1.csv
Read: 202203-citibike-tripdata_2.csv
Read: 202210-citibike-tripdata_1.csv
Read: 202210-citibike-tripdata_2.csv
Read: 202210-citibike-tripdata_3.csv
Read: 202207-citibike-tripdata_1.csv
Read: 202207-citibike-tripdata_3.csv
Read: 202207-citibike-tripdata_4.csv
Read: 202207-citibike-tripdata_2.csv
Read: 202202-citibike-tripdata_2.csv
Read: 202202-citibike-tripdata_1.csv
Read: 202212-citibike-tripdata_2.csv
Read: 202212-citibike-tripdata_1.csv
Read: 202206-citibike-tripdata_3.csv
Read: 202206-citibike-tripdata_2.csv
Read: 202206-citibike-tripdata_4.csv
Read: 202206-citibike-tripdata_1.csv
Read: 202208-citibike-tripdata_4.csv
R

In [6]:
#understanding the dataframe:
df.shape

(29838806, 10)

In [7]:
df.head()

,rideable_type,started_at,ended_at,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual
0,classic_bike,2022-01-13 21:36:47.689,2022-01-13 21:46:02.024,5 Ave & E 63 St,Broadway & W 51 St,40.766368,-73.971518,40.762288,-73.983362,member
1,classic_bike,2022-01-16 17:56:23.889,2022-01-16 18:03:50.269,Grand Army Plaza & Plaza St West,Bedford Ave & Montgomery St,40.672968,-73.970880,40.665816,-73.956934,member
2,electric_bike,2022-01-18 07:10:04.799,2022-01-18 07:20:54.450,W 20 St & 10 Ave,Broadway & W 51 St,40.745686,-74.005141,40.762288,-73.983362,member
3,classic_bike,2022-01-22 12:10:10.225,2022-01-22 12:20:06.899,W 54 St & 9 Ave,10 Ave & W 28 St,40.765849,-73.986905,40.750664,-74.001768,member
4,classic_bike,2022-01-08 16:35:16.497,2022-01-08 16:45:33.279,Sharon St & Olive St,Driggs Ave & Lorimer St,40.715353,-73.938560,40.721791,-73.950415,casual


In [8]:
df.tail()

,rideable_type,started_at,ended_at,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual
29838801,classic_bike,2022-04-18 17:26:38.829,2022-04-18 17:40:48.908,W 4 St & 7 Ave S,8 Ave & W 31 St,40.734011,-74.002939,40.750585,-73.994685,casual
29838802,classic_bike,2022-04-18 13:51:46.012,2022-04-18 13:57:06.613,Bedford Ave & Bergen St,Bergen St & Vanderbilt Ave,40.676368,-73.952918,40.679439,-73.968044,casual
29838803,electric_bike,2022-04-22 16:31:28.973,2022-04-22 16:37:48.641,Plaza St East & Flatbush Ave,DeKalb Ave & Vanderbilt Ave,40.673134,-73.969106,40.689407,-73.968855,casual
29838804,electric_bike,2022-04-02 14:02:50.230,2022-04-02 14:12:16.068,W 45 St & 8 Ave,E 41 St & Madison Ave (SW corner),40.759291,-73.988597,40.752165,-73.979922,casual
29838805,classic_bike,2022-04-14 20:55:09.859,2022-04-14 21:05:23.745,Bergen St & Smith St,Bergen St & Vanderbilt Ave,40.686744,-73.990632,40.679439,-73.968044,casual


### Getting weather data using NOAA's API

##### Some notes: 
- I downloaded the weather dataset from NOAA's website because it was easier, since in Kaggle it isn't possible to use API. I'll upload it here as a csv.
- There was no TAVG, so I calculated it and created a new column for that purpose.

### If API would work, this is how I'd do it:

````python
#defining my NOAA token:
Token = 'PRqSNMTFOrNOzcjYCmZrbQQGnZnKMGoc'

````python
r = requests.get('https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&datatypeid=TMAX&datatypeid=TMIN&datatypeid=TAVG&limit=1000&stationid=GHCND:USW00014732&startdate=2022-01-01&enddate=2022-12-31&units=metric', headers={'token': 'PRqSNMTFOrNOzcjYCmZrbQQGnZnKMGoc'})

d = json.loads(r.text)

````python
d = r.json()

#split TMAX e TMIN
tmax = [item for item in d['results'] if item['datatype'] == 'TMAX']
tmin = [item for item in d['results'] if item['datatype'] == 'TMIN']

#extract dates and values
dates = [item['date'] for item in tmax]
temps_max = [item['value'] for item in tmax]
temps_min = [item['value'] for item in tmin]

#calculate average
temps_avg = [(mx + mn) / 2 for mx, mn in zip(temps_max, temps_min)]

```python
print(len(dates), len(temps_avg))

````python
from datetime import datetime
#putting the results in a dataframe, converting it to proper formats; 
df_temp = pd.DataFrame()
df_temp['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates]
df_temp['avgTemp'] = temps_avg

### Uploading CSV from NOAA:
 - Kaggle doesn't allow access to other websites by default. So let's upload this in another way, by downloading the weather dataset directly from NOAA and uploading it here:

In [9]:
#lets upload the csv weather file to the notebook:
df_temp = pd.read_csv(r"/kaggle/input/datasets/danielabranca/data-weather/weather_ny_2022.csv")

In [10]:
df_temp.head()

,STATION,NAME,DATE,TAVG,TMAX,TMIN
0,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-01,NaN,57,50
1,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-02,NaN,60,39
2,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-03,NaN,39,24
3,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-04,NaN,36,21
4,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-05,NaN,48,32


In [11]:
#calculating TAVG and standardizing all variables with ºC and round 0:
df_temp['TMAX'] = ((df_temp['TMAX'] - 32) * 5/9).round(0).astype(int)
df_temp['TMIN'] = ((df_temp['TMIN'] - 32) * 5/9).round(0).astype(int)
df_temp['TAVG'] = (((df_temp['TMAX'] + df_temp['TMIN']) / 2 - 32) * 5/9).round(0).astype(int)
df_temp.head()

,STATION,NAME,DATE,TAVG,TMAX,TMIN
0,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-01,-11,14,10
1,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-02,-12,16,4
2,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-03,-18,4,-4
3,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-04,-19,2,-6
4,USW00014732,"LAGUARDIA AIRPORT, NY US",2022-01-05,-15,9,0


In [12]:
#checking string types:
df_temp.dtypes

STATION    object
NAME       object
DATE       object
TAVG        int64
TMAX        int64
TMIN        int64
dtype: object

In [13]:
df.dtypes

rideable_type          object
started_at             object
ended_at               object
start_station_name     object
end_station_name       object
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual          object
dtype: object

In [14]:
#converting started_at and ended_at to datetime:
df['started_at'] = pd.to_datetime(df['started_at'], format='mixed')
df['ended_at'] = pd.to_datetime(df['ended_at'], format='mixed')

In [15]:
#deriving start_date column with date only:
df['start_date'] = df['started_at'].dt.date
df['start_date'] = pd.to_datetime(df['start_date'])

In [16]:
df.dtypes

rideable_type                 object
started_at            datetime64[ns]
ended_at              datetime64[ns]
start_station_name            object
end_station_name              object
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                 object
start_date            datetime64[ns]
dtype: object

In [17]:
#converting date from df into datetime as well:
df_temp['DATE'] = pd.to_datetime(df_temp['DATE'])

In [18]:
df_temp.dtypes

STATION            object
NAME               object
DATE       datetime64[ns]
TAVG                int64
TMAX                int64
TMIN                int64
dtype: object

In [19]:
#doing the merge:
df_merged = df.merge(df_temp, how='left', left_on='start_date', right_on='DATE', indicator =True)
print(df_merged['_merge'].value_counts())

_merge
both          29838166
left_only          640
right_only           0
Name: count, dtype: int64


In [ ]:
#seeing lines with no correspondence (even though it's neglectible):
df_missing = df_merged[df_merged['_merge'] == 'left_only']
print(df_missing['start_date'].value_counts())

In [20]:
#saving this as pq file:
df_merged.to_parquet('/kaggle/working/citibike_merged_2022.parquet', index=False)

### Pulling files to git hub repo:

In [ ]:
#adding to github repo
!git add citybike_merge.ipynb
!git commit -m "Add notebook on API data access and merging"
!git push -u origin main